In [2]:
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


class PFATracker:
    def __init__(self, concepts, num_levels=6, top_k=5, alpha=0.1):
        """
        concepts: list of concept strings
        num_levels: Bloom levels (default 6)
        top_k: neighbors for propagation
        alpha: propagation strength
        """
        self.concepts = concepts
        self.num_levels = num_levels
        self.top_k = top_k
        self.alpha = alpha

        self.num_concepts = len(concepts)

        # Map concept -> index
        self.idx = {c: i for i, c in enumerate(concepts)}

        # Success / failure counts
        self.successes = np.zeros((self.num_concepts, num_levels))
        self.failures = np.zeros((self.num_concepts, num_levels))

        # -------------------------
        # Parameters (tunable)
        # -------------------------

        # Concept bias (can stay 0 initially)
        self.beta_concept = np.zeros(self.num_concepts)

        # Level difficulty (harder → more negative)
        self.beta_level = np.array([0.5, 0.2, 0.0, -0.3, -0.6, -1.0])

        # Learning rates per level
        self.gamma_level = np.array([0.4, 0.35, 0.3, 0.25, 0.2, 0.15])

        # Failure penalty per level
        self.rho_level = np.array([0.3, 0.35, 0.4, 0.45, 0.5, 0.6])

        # -------------------------
        # Build similarity graph
        # -------------------------
        self._build_similarity_graph()

    def _build_similarity_graph(self):
        model = SentenceTransformer("BAAI/bge-base-en-v1.5")

        embeddings = model.encode(
            self.concepts,
            normalize_embeddings=True
        )

        sim_matrix = embeddings @ embeddings.T  # cosine similarity

        self.neighbors = []

        for i in range(self.num_concepts):
            sims = sim_matrix[i]
            idxs = np.argsort(-sims)[1:self.top_k + 1]  # exclude self
            self.neighbors.append(idxs)

        self.sim_matrix = sim_matrix

    # -------------------------
    # Prediction
    # -------------------------
    def predict(self, concept, level):
        i = self.idx[concept]
        k = level - 1

        z = (
            self.beta_concept[i]
            + self.beta_level[k]
            + self.gamma_level[k] * self.successes[i][k]
            - self.rho_level[k] * self.failures[i][k]
        )

        return sigmoid(z)

    # -------------------------
    # Update + propagation
    # -------------------------
    def update(self, concept, level, correct):
        i = self.idx[concept]
        k = level - 1

        old_p = self.predict(concept, level)

        # Update counts
        if correct:
            self.successes[i][k] += 1
        else:
            self.failures[i][k] += 1

        new_p = self.predict(concept, level)

        delta = new_p - old_p

        # Clip to avoid instability
        delta = np.clip(delta, -0.1, 0.1)

        # Propagate to neighbors
        for j in self.neighbors[i]:
            sim = self.sim_matrix[i][j]

            for lvl in range(self.num_levels):
                self._propagate(j, lvl, delta, sim)

    def _propagate(self, j, lvl, delta, sim):
        """
        Soft propagation by adjusting counts slightly
        """
        weight = self.alpha * sim / (lvl + 1)  # decay for higher levels

        if delta > 0:
            self.successes[j][lvl] += weight
        else:
            self.failures[j][lvl] += weight

    # -------------------------
    # Utility
    # -------------------------
    def get_concept_mastery(self, concept):
        """
        Returns average mastery across levels
        """
        i = self.idx[concept]
        probs = [
            self.predict(concept, lvl + 1)
            for lvl in range(self.num_levels)
        ]
        return float(np.mean(probs))

    def get_state_vector(self):
        """
        Flattened state for RL:
        [P(c1,l1), P(c1,l2), ..., P(cN,l6)]
        """
        state = []
        for c in self.concepts:
            for lvl in range(1, self.num_levels + 1):
                state.append(self.predict(c, lvl))
        return np.array(state)

In [5]:
concepts = [
    "binary search tree",
    "heap data structure",
    "sorting algorithm",
    "hash table",
    "linked list"
]

tracker = PFATracker(concepts)

# Simulate interaction
print("bst")
print(tracker.predict("binary search tree", 2))
print("heap")
print(tracker.predict("heap data structure", 2))
print("sort")
print(tracker.predict("sorting algorithm", 2))
print("hash")
print(tracker.predict("hash table", 2))

tracker.update("binary search tree", level=2, correct=True)

#print(tracker.predict("binary search tree", 2))

# Check propagation effect
print("after update 1")
print("bst")
print(tracker.predict("binary search tree", 2))
print("heap")
print(tracker.predict("heap data structure", 2))
print("sort")
print(tracker.predict("sorting algorithm", 2))
print("hash")
print(tracker.predict("hash table", 2))
print("linked list")
print(tracker.predict("linked list", 2))

tracker.update("linked list", level=2, correct=True)

print("after update 2")
print("bst")
print(tracker.predict("binary search tree", 2))
print("heap")
print(tracker.predict("heap data structure", 2))
print("sort")
print(tracker.predict("sorting algorithm", 2))
print("hash")
print(tracker.predict("hash table", 2))
print("linked list")
print(tracker.predict("linked list", 2))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


bst
0.549833997312478
heap
0.549833997312478
sort
0.549833997312478
hash
0.549833997312478
after update 1
bst
0.6341355910108007
heap
0.5527793030137121
sort
0.5528461022222357
hash
0.55271899317395
linked list
0.5526473083719012
after update 2
bst
0.6367701008686569
heap
0.55565391621791
sort
0.5554103815435173
hash
0.5553795839563351
linked list
0.6367701008686569
